# Lending Club Loan Performance & Risk Analysis (2007–2018)
## Stage 4 — Power BI Data Export

## Introduction

This notebook is the **final stage** of the pipeline. It reads the cleaned loan dataset and produces a set of **pre-aggregated summary tables** in CSV format, ready to be imported directly into Power BI for interactive dashboard creation.

Exporting aggregated tables (rather than the full 1.37M row dataset) keeps Power BI responsive and makes data relationships explicit.

**Input:** `../data/cleaned_data.csv` — produced by `01_data_cleaning.ipynb`.

**Output:** Seven CSV files saved to `../data/powerbi/` — import these into Power BI.

See `../powerbi/dashboard_notes.md` for the recommended dashboard layout, visuals, and KPI cards.

---

## Contents

1. **Setup & Data Loading**
2. **Table 1 — Portfolio Summary KPIs**
3. **Table 2 — Default Rate by Grade**
4. **Table 3 — Default Rate & Profit by Purpose**
5. **Table 4 — Default Rate by Issue Year (Vintage)**
6. **Table 5 — Default Rate by FICO Band**
7. **Table 6 — Default Rate by Loan Term**
8. **Table 7 — Default Rate by DTI Band**
9. **Export Confirmation**

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import os

# Create output directory if it doesn't exist
os.makedirs("../data/powerbi", exist_ok=True)

df = pd.read_csv("../data/cleaned_data.csv")

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print(f"Default rate: {df['default'].mean()*100:.2f}%")

---

## 2. Table 1 — Portfolio Summary KPIs

Top-level KPIs for the Power BI summary / overview page.

In [ ]:
portfolio_summary = pd.DataFrame([{
    "total_loans":        len(df),
    "total_defaults":     int(df["default"].sum()),
    "default_rate_pct":   round(df["default"].mean() * 100, 2),
    "total_funded_M":     round(df["loan_amnt"].sum() / 1e6, 2),
    "avg_loan_amnt":      round(df["loan_amnt"].mean(), 2),
    "avg_int_rate_pct":   round(df["int_rate"].mean(), 2),
    "avg_fico_score":     round(df["fico_score"].mean(), 1),
    "total_profit_M":     round((df["total_pymnt"] - df["loan_amnt"]).sum() / 1e6, 2),
}])

portfolio_summary.to_csv("../data/powerbi/01_portfolio_summary.csv", index=False)
portfolio_summary

---

## 3. Table 2 — Default Rate by Credit Grade

Used for bar charts and risk-tier visuals in Power BI.

In [ ]:
by_grade = (
    df.groupby("grade", observed=False)
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum"),
        avg_int_rate=("int_rate", "mean"),
        avg_loan_amnt=("loan_amnt", "mean"),
        profit=("total_pymnt", lambda x: (x - df.loc[x.index, "loan_amnt"]).sum())
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        pct_of_portfolio=lambda x: (x["loans"] / x["loans"].sum() * 100).round(1),
        avg_int_rate=lambda x: x["avg_int_rate"].round(2),
        avg_loan_amnt=lambda x: x["avg_loan_amnt"].round(0),
        profit=lambda x: x["profit"].round(2)
    )
    .reset_index()
)

by_grade.to_csv("../data/powerbi/02_defaults_by_grade.csv", index=False)
by_grade

---

## 4. Table 3 — Default Rate & Profit by Purpose

Used for purpose-level risk and profitability visuals.

In [ ]:
by_purpose = (
    df.groupby("purpose_group", observed=False)
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum"),
        profit=("total_pymnt", lambda x: (x - df.loc[x.index, "loan_amnt"]).sum())
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        pct_of_portfolio=lambda x: (x["loans"] / x["loans"].sum() * 100).round(1),
        profit=lambda x: x["profit"].round(2)
    )
    .sort_values("default_rate_pct", ascending=False)
    .reset_index()
)

by_purpose.to_csv("../data/powerbi/03_defaults_by_purpose.csv", index=False)
by_purpose

---

## 5. Table 4 — Default Rate by Issue Year (Vintage)

Used for time-series / line charts tracking portfolio trends.

In [ ]:
by_year = (
    df.groupby("issue_year")
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum"),
        avg_loan_amnt=("loan_amnt", "mean"),
        avg_int_rate=("int_rate", "mean")
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        avg_loan_amnt=lambda x: x["avg_loan_amnt"].round(0),
        avg_int_rate=lambda x: x["avg_int_rate"].round(2)
    )
    .reset_index()
)

by_year.to_csv("../data/powerbi/04_defaults_by_year.csv", index=False)
by_year

---

## 6. Table 5 — Default Rate by FICO Band

Used for credit-score tier visuals.

In [ ]:
fico_bins  = [0, 600, 700, 750, 900]
fico_labels = ["Poor (<600)", "Fair (600-699)", "Good (700-749)", "Excellent (750+)"]

df["fico_band"] = pd.cut(df["fico_score"], bins=fico_bins, labels=fico_labels)

by_fico = (
    df.groupby("fico_band", observed=False)
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum")
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        pct_of_portfolio=lambda x: (x["loans"] / x["loans"].sum() * 100).round(1),
        sort_order=[1, 2, 3, 4]
    )
    .reset_index()
)

df.drop(columns=["fico_band"], inplace=True)

by_fico.to_csv("../data/powerbi/05_defaults_by_fico.csv", index=False)
by_fico

---

## 7. Table 6 — Default Rate by Loan Term

In [ ]:
by_term = (
    df.groupby("term")
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum"),
        avg_int_rate=("int_rate", "mean")
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        pct_of_portfolio=lambda x: (x["loans"] / x["loans"].sum() * 100).round(1),
        avg_int_rate=lambda x: x["avg_int_rate"].round(2),
        term_label=lambda x: x.index.map({36: "36 Months", 60: "60 Months"})
    )
    .reset_index()
)

by_term.to_csv("../data/powerbi/06_defaults_by_term.csv", index=False)
by_term

---

## 8. Table 7 — Default Rate by DTI Band

In [ ]:
dti_bins   = [0, 10, 20, 30, df["dti"].max() + 1]
dti_labels = ["Low (<10)", "Medium (10-20)", "High (20-30)", "Very High (30+)"]

df["dti_band"] = pd.cut(df["dti"], bins=dti_bins, labels=dti_labels)

by_dti = (
    df.groupby("dti_band", observed=False)
    .agg(
        loans=("default", "count"),
        defaults=("default", "sum")
    )
    .assign(
        default_rate_pct=lambda x: (x["defaults"] / x["loans"] * 100).round(2),
        pct_of_portfolio=lambda x: (x["loans"] / x["loans"].sum() * 100).round(1),
        sort_order=[1, 2, 3, 4]
    )
    .reset_index()
)

df.drop(columns=["dti_band"], inplace=True)

by_dti.to_csv("../data/powerbi/07_defaults_by_dti.csv", index=False)
by_dti

---

## 9. Export Confirmation

In [ ]:
import os

export_dir = "../data/powerbi"
files = sorted(os.listdir(export_dir))

print(f"Export complete. {len(files)} files saved to {export_dir}/\n")
for f in files:
    path = os.path.join(export_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

print("\nNext step: Open Power BI Desktop → Get Data → Text/CSV → select each file above.")
print("See powerbi/dashboard_notes.md for the recommended dashboard design.")